In [2]:
import yaml
import os
import re
import requests

yml_path = "../resources/scadsai.yml"

In [4]:
# Load YAML
with open(yml_path, "r", encoding="utf-8") as f:
    data = yaml.safe_load(f)

api_key = os.environ.get("ZENODO_API_KEY")
headers = {"Authorization": f"Bearer {api_key}"} if api_key else {}

def extract_zenodo_id(urls):
    for u in urls:
        if u.startswith("https://zenodo.org/"):
            match = re.search(r"records/(\d+)", u)
            if match:
                return match.group(1)
    return None

for record in data.get("resources", []):
    if "num_files" in record:
        continue

    urls = record.get("url", [])
    if isinstance(urls, str):
        urls = [urls]

    zenodo_id = extract_zenodo_id(urls)
    if zenodo_id is None:
        continue

    api_url = f"https://zenodo.org/api/records/{zenodo_id}"
    try:
        resp = requests.get(api_url, headers=headers)
        resp.raise_for_status()
        record_json = resp.json()
        files = record_json.get("files", [])
        record["num_files"] = len(files)
        print(f"Record {record.get('name')} (id {zenodo_id}) contains {len(files)} files")
    except Exception as e:
        print(f"Failed for record {record.get('name')} (id {zenodo_id}): {e}")
    

    # Save back
    with open(yml_path, "w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, allow_unicode=True, sort_keys=False)

Record Smart Slide Generation: Re-Using Training Materials Through the Power of LLMs (id 17541532) contains 1 files
Record Selected meta data of accepted contributions to some AI/ML conferences 2020-2025 (id 17478160) contains 4 files
Record How Generative AI impacts research and teaching (id 17422917) contains 4 files
Record TreeSway (id 17240205) contains 10 files
Record AI4Psychology KI-Kompetenz-Training (id 17203260) contains 6 files
Record Research Data Management (id 17186869) contains 2 files
Record ScaDS.AI Meetup #8 2025: Workshop Model Context Protocol (MCP) (id 17158770) contains 1 files
Record ScaDS.AI Meetup #8 2025: AI Insights Model Context Protocol (MCP) (id 17158643) contains 1 files
Record Data literacy, Open Science and Collaborative Coding and CI/CD - Slides for the ScaDS.AI GA-Workshop "Working with data and code like a pro" (id 17130780) contains 2 files
Record Explainable Machine Learning (id 17116758) contains 2 files
Record Prompt Engineering Techniques - Slid